In [ ]:
import pandas as pd

#  ann-thyroid.names describes 21 attributes (15 binary + 6 continuous) + class in last column
col_names = [f"attr_{i}" for i in range(1, 22)] + ["class"]

train_df = pd.read_csv(
    r"C:\Users\USER\Desktop\Basic codes in the WQU\thyroid classification project\thyroid dataset\thyroid+disease\ann-train.data",
    sep=r"\s+", header=None, names=col_names
)

test_df = pd.read_csv(
    r"C:\Users\USER\Desktop\Basic codes in the WQU\thyroid classification project\thyroid dataset\thyroid+disease\ann-test.data",
    sep=r"\s+", header=None, names=col_names
)

print(train_df.shape, test_df.shape)
train_df["class"].value_counts()

: 

In [13]:
train_df.head()

,attr_1,attr_2,attr_3,attr_4,attr_5,attr_6,attr_7,attr_8,attr_9,attr_10,...,attr_13,attr_14,attr_15,attr_16,attr_17,attr_18,attr_19,attr_20,attr_21,class
0,0.73,0,1,0,0,0,0,0,1,0,...,0,0,0,0,0.00060,0.015,0.120,0.082,0.146,3
1,0.24,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0.00025,0.030,0.143,0.133,0.108,3
2,0.47,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0.00190,0.024,0.102,0.131,0.078,3
3,0.64,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0.00090,0.017,0.077,0.090,0.085,3
4,0.23,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0.00025,0.026,0.139,0.090,0.153,3


In [12]:
print(train_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 3772 entries, 0 to 3771
Data columns (total 22 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   attr_1   3772 non-null   float64
 1   attr_2   3772 non-null   int64  
 2   attr_3   3772 non-null   int64  
 3   attr_4   3772 non-null   int64  
 4   attr_5   3772 non-null   int64  
 5   attr_6   3772 non-null   int64  
 6   attr_7   3772 non-null   int64  
 7   attr_8   3772 non-null   int64  
 8   attr_9   3772 non-null   int64  
 9   attr_10  3772 non-null   int64  
 10  attr_11  3772 non-null   int64  
 11  attr_12  3772 non-null   int64  
 12  attr_13  3772 non-null   int64  
 13  attr_14  3772 non-null   int64  
 14  attr_15  3772 non-null   int64  
 15  attr_16  3772 non-null   int64  
 16  attr_17  3772 non-null   float64
 17  attr_18  3772 non-null   float64
 18  attr_19  3772 non-null   float64
 19  attr_20  3772 non-null   float64
 20  attr_21  3772 non-null   float64
 21  class    3772 non-null   

In [ ]:
train_label = train_df.iloc[:, -1]

train_label.head()

0    3
1    3
2    3
3    3
4    3
Name: class, dtype: int64

In [ ]:
test_df.head()

In [ ]:
test_label = test_df.iloc[:, -1]

test_label.head()

In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import torch
from torch.utils.data import TensorDataset, DataLoader


le = LabelEncoder()
y_encoded = le.fit_transform(y_label)
class_names = le.classes_

AttributeError: partially initialized module 'torch' has no attribute 'nn' (most likely due to a circular import)

In [ ]:
# Standardized features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(train_df)

: 

In [ ]:
# convert to pytorch tensors
X_train_tensor = torch.tensor(df_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_label, dtype=torch.float32)

X_test_tensor = torch.tensor(df_test, dtype=torch.long)
y_test_tensor = torch.tensor(test_label, dtype=torch.long)

In [ ]:
# wrap in dataloaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
print("Train set:", X_train_tensor.shape)
print("Test set:", X_test_tensor.shape)
print("Number of Classes:", len(class_names))
print("Class Names:", class_names)

In [ ]:
# model definition

import torch.nn as nn


class ModelMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(ModelMLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

        def forward(self, x):
            return self.model(x)

# Instantiate the model
input_dim = X_train_tensor.shape[1]     # for the features
hiddn_dim = 16
output_dim = len(class_names)


model = ModelMLP(input_dim, output_dim)


# display model structure
print(model)

In [ ]:
# FULL BATCH GRADIENT DESCENT

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time


# set seed for reproducibility
torch.manual_seed(42)

# Model from earlier section (assumed defined)
model_gd = ModelMLP(input_dim, hidden_dim, output_dim)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)


# store loss and accuracy per epoch
train_loss_gd = []
train_acc_gd = []

# training loop  --> Full Batch Gradient Descent
n_epoch = 100

start_time = time.perf_counter()

for epoch in range(n_epochs):
    # set the model to training mode
    model_gd.train()

    # forward pass on the whole training set
    logits = model(X_train_tensor)
    loss = loss_fn(logits, y_train_tensor)

    # backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


    # store loss
    train_loss_gd.append(loss.item())

    # compute accuracy
    preds = torch.argmax(logits, dim=1)
    acc = (preds == y_train_tensor).float().mean().item()
    train_acc_gd.append(acc)

    # print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:>2} | Loss: {loss.item():.4f} | Accuracy: {acc*100:.2f}%")

end_time = time.perf_counter()
print(f"Training completed in {end_time - start_time:.2f} seconds")

In [ ]:
# STOCHASTIC GRADIENT DESCENT
import torch
import torch.nn as nn
import torch.optim as optim
import random


# set seed for reproducibility
torch.manual_seed(42)

# create a nrew model instance
model_sgd = MLPModel(input_dim=8, hidden_dim=16, output_dim=3)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_sgd.parameters(), lr=0.01)


# track metrics
sgd_loss_values = []
sgd_accuracy_values = []


# convert train set into list of samples
train_samples = list(zip(X_train_tensor, y_train_tensor))

start_time = time.perf_counter()

# Training loop: SGD
for epoch in range(100):
    epoch_loss = 0
    correct = 0

    # shuffle training samples
    torch.random.manual_seed(epoch)    # ensures reproducibility across epochs
    shuffled_samples = train_samples.copy()
    random.shuffle(shuffled_samples)

    for x_samples, y_samples in shuffled_samples:
        x_sample = x_sample.unsqueeze(0)   # add batch dim
        y_sample = y_sample.unsqueeze(0)


        # forward pass
        outputs = model_sgd(x_sample)
        loss = criterion(outputs, y_sample)

        # backpropagation and update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


        # track loss and accuracy
        epoch += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == y_sample).sum().item()

    avg_loss = epoch_loss / len(train_samples)
    accuracy = correct / len(train_samples)

    sgd_loss_values.append(avg_loss)
    sgd_accuracy_values.append(accuracy)

    if(epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:>2} | Loss: {avg_loss:.4f} | Accuracy: {accuracy*100:.2f}%")

end_time = time.perf_counter()
print(f"Training completed in {end_time - start_time:.2f} seconds")

In [ ]:
# MINI BATCH GRADIENT DESCENT
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim


# define hyperparameters
batch_size = 32
epochs = 100
learning_rate = 0.01

# initialize model and optimizer
model_mb = ModelMLP(input_dim, hidden_dim, output_dim)
optimizer_mb = optim.SGD(model_md.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()


# create DataLoader for mini-batch SGD
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# store metrics
losses_mb = []
acc_mb = []


start_time = time.perf_counter()

# training loop
for epoch in range(epochs):
    model_m.train()
    epoch_loss = 0.0
    correct = 0
    total = 0


    for X_batch, y_batch in train_loader:
        # forward pass 
        outputs = model_mb(X_batch)
        loss = criterion(outputs, y_batch)

        # backward pass
        optimizer_mb.zero_grad()
        loss.backward()
        optimizer_mb.step()

        # track loss
        epoch_loss += loss.item() * X_batch.size(0)

        # track accuracy
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)

    avg_loss = epoch_loss / total
    acc = correct / total

    losses_mb.append(avg_loss)
    accs_mb.append(acc)

    print(f"[Mini-Batch] Epoch {epoch+1:2d}: Loss: {avg_loss:.4f} | Accuracy = {acc:.4f}")

    end_time = time.perf_counter()
    print(f"Training completed in {end_time - start_time:.2f} seconds")

In [ ]:
# compare the three optimizers {full batch GD, Stochastic GD, and Mini-Batch GD}

import matplotlib.pyplot as plt

epochs = len(train_loss_gd)     # assuming all lists are same length

plt.figure(figsize=(14, 5))

# plot loss
plt.subplot(1, 2, 1)
plt.plot(range(epochs), train_loss_gd, label="GD (Batch)", linewidth=2)
plt.plot(range(epochs), sgd_loss_values, label="SGD (per sample)", linewidth=2)
plt.plot(range(epochs), losses_mb, label="Mini-Batch", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Convergence")
plt.legend()
plt.show()




# plot the accuracy too
plt.subplot(1, 2, 2)
plt.plot(range(epochs), train_acc_gd, label="GD (Batch)", linewidth=2)
plt.plot(range(epochs), sgd_accuracy_values, label="SGD (Per Sample)", linewidth=2)
plt.plot(range(epochs), accs_mb, label="Mini-batch", linewidth=2)
plt.xlabel("Epoch")
plt.yabel("Accuracy")
plt.title("Accuracy Over Epochs")
plt.legend()
plt.grid(True)


plt.tight_layout()
plt.show()

In [ ]:
# Evaluation on Test Set 

from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

def evaluate_model(model, X_test, y_test, class_names, title):
    # put model in eval mode
    model.eval()

    # disable gradient tracking
    with torch.no_grad():
        outputs = model(X_test)
        _, preds = torch.max(outputs, 1)   # get predicted class indices

    # convert to numpy for sklearn
    y_true = y_test.cpu().numpy()
    y_pred = preds.cpu().numpy()

    # accuracy
    acc = accuracy_score(y_true, y_pred)
    print(f"Test Accuracy ({title}): {acc:.4f}")


    # confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(10, 8))
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=45)
    plt.title(f"Confusion Matrix - {title}")
    plt.show()

# Evaluate all models
evaluate_model(model_gd, X_test_tensor, y_test_tensor, class_names, "Gradient Descent (Batch)")
evaluate_model(model_sgd, X_test_tensor, y_test_tensor, class_names, "Stochastic Gradient Descent (per sample)")
evaluate_model(model_mb, X_test_tensor, y_test_tensor, class_names, "Mini-Batch SGD")